<h1>ENSO Index - Ensemble Means</h1>

![UFS-logo](../../../UFS-Logo-RGB-2csolidshorizontal-72dpi-min.png)

In [1]:
basedir = f'../../../..'

In [2]:
import os
import sys
import xarray as xr

# Point to root directory of repository
root_dir = os.path.join(os.getcwd(), basedir)
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)

from src.datareader import datareader as dr
from src.util import util, stats

import warnings
warnings.filterwarnings('ignore')

<h5>Get data readers</h5>

In [3]:
ufs_vars_list = ['tmpsfc', 'tsfc']
era5_var = 'sea_surface_temperature'
ecmwf_var = 'sst'

In [4]:
ufs_experiments = ['baseline', 'beta.0.1', 'c96_beta.0.1', 'cpc_ics']

In [5]:
# Location of .nc file
ecmwf_hindcasts_file = '/home/thamzey/sst-seasonal-monthly-single-levels.nc'
ecmwf_hindcast_ds = xr.open_dataset(ecmwf_hindcasts_file, engine='netcdf4')

# Rename dimensions/coordinates to match our init+lead paradigm.
ecmwf_hindcast_ds = ecmwf_hindcast_ds.rename_dims({'forecast_reference_time': 'init',
                                                   'forecastMonth': 'lead'})

ecmwf_hindcast_ds = ecmwf_hindcast_ds.rename({'forecast_reference_time': 'init',
                                              'forecastMonth': 'lead'})

# Wrap it up into a DataReader object.
ecmwf_data_reader = dr.getDataReader(datasource='SUPPLIED', dataset=ecmwf_hindcast_ds)

In [6]:
era5_data_reader = dr.getDataReader(datasource='ERA5')

No filename provided; deferring to default
Reading data from gs://gcp-public-data-arco-era5/ar/1959-2022-6h-512x256_equiangular_conservative.zarr


In [7]:
era5_data_reader.describe(era5_var)


Variable: sea_surface_temperature
Dimensions: ('time', 'lat', 'lon')
Shape: (92044, 256, 512)
Attributes:
  - long_name: Sea surface temperature
  - short_name: sst
  - units: K


In [8]:
# ecmwf_data_reader.describe(ecmwf_var)

<h5>Define time period</h5>

In [9]:
time_range = ("2017-02-01","2021-12-31T23")
initmonths = (11,)

<h5>Define nino 3.4 region</h5>

In [10]:
region = {
    'latmin': -5.0,
    'latmax': 5.0,
    'lonmin': 190.0,
    'lonmax':240.0
}

<h5>Get the monthly climatology for nino 3.4</h5>

In [11]:
%%capture captured_output
# First, collect all the ensemble means for the UFS models.
ufs_ds = util.combine_ufs_means(ufs_experiments, ufs_vars_list, time_range, region=region, initmonths=initmonths)

<h5>Get the corresponding ERA5 data</h5>

In [12]:
era5_ds = era5_data_reader.retrieve(var=era5_var,
                lat=(region['latmin'], region['latmax']),
                lon=(region['lonmin'], region['lonmax']),
                time=time_range)

In [13]:
ecmwf_ds = ecmwf_data_reader.retrieve(var=ecmwf_var,
                lat=(region['latmin'], region['latmax']),
                lon=(region['lonmin'], region['lonmax']),
                initmonths=initmonths, time=time_range)

In [14]:
# Because both the ecmwf and ufs models have init+lead structure,
# combine ecmwf into the ufs dataset as a standalone model.

# Rename to match ufs
ecmwf_ds = ecmwf_ds.rename({'sst': 'tmpsfc'})
# consider ecmwf a 'member'
ecmwf_ds = ecmwf_ds.assign_coords(member=('member', ['ecmwf']))
# Concatenate with ufs
ufs_ds = xr.concat([ufs_ds, ecmwf_ds], dim='member', coords='minimal', compat='equals')

<h5>Calculate climatology (this may take a couple minutes)</h5>

In [15]:
ufs_stats = stats.calc_climatology_anomaly(ufs_ds, area_mean=True, use_member_climatology=True)

In [ ]:
era5_stats = stats.calc_climatology_anomaly(era5_ds, area_mean=True, use_member_climatology=True)

In [ ]:
stats.plot_index_spaghetti(ufs_stats=ufs_stats,
                           verif_stats=era5_stats,
                           calc_anomaly=True,
                           title=f'Nino3.4 SST',
                           verif_label='ERA5 SST',
                           dpi=300)